# 🧠 Robotic Arm Quality Control Training (Custom CNN)
Ensure you have connected to a **T4 GPU** (Runtime > Change runtime type).


In [ ]:
import os

# Configure Kaggle API Token
os.environ['KAGGLE_API_TOKEN'] = "KGAT_62c6681f4e466f62ad1b06ae3f2db98a"

# Install Kagglehub
!pip install -q kagglehub

import kagglehub
import shutil
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import matplotlib.pyplot as plt
import zipfile

print("✅ Dependencies loaded and API set!")


In [ ]:
print("📥 Downloading Fruits 360 (for clean/good fruits)...")
path_fruits360 = kagglehub.dataset_download("moltean/fruits")
print(f"Fruits 360 path: {path_fruits360}")


In [ ]:
# 🛑 STOP: Ensure you have uploaded the Mendeley ZIP file to Colab before running this cell!
zip_files = [f for f in os.listdir('.') if f.endswith('.zip')]

if zip_files:
    zip_path = zip_files[0]
    print(f"📦 Found uploaded zip: {zip_path}. Unzipping...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall("mendeley_dataset")
    print("✅ Mendeley dataset unzipped successfully!")
else:
    raise FileNotFoundError("❌ No ZIP file found! Please upload the Mendeley ZIP to the left sidebar in Colab.")


In [ ]:
# Create final structure
base_dir = './dataset'
good_dir = os.path.join(base_dir, 'good')
defective_dir = os.path.join(base_dir, 'defective')

if os.path.exists(base_dir):
    shutil.rmtree(base_dir)

os.makedirs(good_dir, exist_ok=True)
os.makedirs(defective_dir, exist_ok=True)

print("📂 Organizing datasets...")

# Filter Fruits 360 (Good) - grab Cherry 1 and Strawberry
f360_train = None
for root, dirs, files in os.walk(path_fruits360):
    if "Training" in dirs:
        f360_train = os.path.join(root, "Training")
        break

if f360_train:
    for fruit in ['Cherry 1', 'Strawberry', 'Apple Red 1']:
        fruit_path = os.path.join(f360_train, fruit)
        if os.path.exists(fruit_path):
            images = [f for f in os.listdir(fruit_path) if f.endswith('.jpg')]
            for img in images[:300]: 
                shutil.copy(os.path.join(fruit_path, img), os.path.join(good_dir, f"f360_{fruit}_{img}"))

# Filter Mendeley (ScienceDirect) dataset
mendeley_path = "./mendeley_dataset"
if os.path.exists(mendeley_path):
    for root, dirs, files in os.walk(mendeley_path):
        # The Mendeley dataset has folders named like "freshapples", "rottenapples"
        folder_name = os.path.basename(root).lower()
        images = [f for f in files if f.endswith(('.jpg', '.png'))]
        
        if 'fresh' in folder_name:
            for img in images[:300]:
                shutil.copy(os.path.join(root, img), os.path.join(good_dir, f"mend_{folder_name}_{img}"))
        elif 'rotten' in folder_name:
            for img in images[:800]:
                shutil.copy(os.path.join(root, img), os.path.join(defective_dir, f"mend_{folder_name}_{img}"))

print(f"✅ 'Good' images: {len(os.listdir(good_dir))}")
print(f"✅ 'Defective' images: {len(os.listdir(defective_dir))}")


In [ ]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

# Data Augmentation to prevent model from memorizing backgrounds
datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=30,
    width_shift_range=0.2,
    height_shift_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    validation_split=0.2
)

print("🔄 Setting up data generators...")
train_gen = datagen.flow_from_directory(
    base_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    subset='training'
)

val_gen = datagen.flow_from_directory(
    base_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    subset='validation'
)

print("Class Indices Mapping:", train_gen.class_indices)


In [ ]:
print("🏗️ Building Custom CNN...")
model = models.Sequential([
    layers.Conv2D(32, (3, 3), activation='relu', input_shape=(224, 224, 3)),
    layers.MaxPooling2D(2, 2),
    
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D(2, 2),
    
    layers.Conv2D(128, (3, 3), activation='relu'),
    layers.MaxPooling2D(2, 2),
    
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dense(1, activation='sigmoid') # Binary output
])

model.compile(optimizer='adam', 
              loss='binary_crossentropy', 
              metrics=['accuracy'])

model.summary()


In [ ]:
print("🔥 Starting Training...")
EPOCHS = 15

history = model.fit(
    train_gen,
    epochs=EPOCHS,
    validation_data=val_gen
)


In [ ]:
# Plot accuracy and loss
acc = history.history['accuracy']
val_acc = history.history['val_accuracy']
loss = history.history['loss']
val_loss = history.history['val_loss']

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(acc, label='Training Accuracy')
plt.plot(val_acc, label='Validation Accuracy')
plt.legend()
plt.title('Accuracy')

plt.subplot(1, 2, 2)
plt.plot(loss, label='Training Loss')
plt.plot(val_loss, label='Validation Loss')
plt.legend()
plt.title('Loss')
plt.show()


In [ ]:
from google.colab import files

model.save('robotic_arm_classifier.h5')
print("✅ Model saved! Triggering automatic download to your local machine...")

# This forces your browser to download the file directly to your local computer
files.download('robotic_arm_classifier.h5')
